# MetaARIMA: Meta-Learning for ARIMA Order Selection

MetaARIMA is a two-stage method that uses meta-learning to speed up ARIMA configuration selection:

1. **Meta-training** (offline, once per frequency): train a meta-learner that maps time-series features to promising ARIMA configurations.
2. **Inference** (online, per series): extract features, shortlist configurations via the meta-learner, and select the best by AICc (using successive halving).

This notebook demonstrates both stages on M4 monthly data:
- **Part 1** — Training MetaARIMA from pre-computed metadata
- **Part 2** — Loading a trained model and forecasting a new series

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# !pip install metaforecast catboost

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

from metaforecast.coseal import MetaARIMA
from metaforecast.coseal.metaarima._base import (
    BEST_CATBOOST_PARAMS,
    ORDER_MAX,
    MetaARIMAUtils,
)

## Part 1 — Training MetaARIMA

Training requires two pre-computed datasets (stored under `assets/metadata/metaarima/`):

- **Features** (`features-dev,m4_monthly.csv`): time-series features (via `tsfeatures`) for each series in the M4 monthly corpus.
- **Scores** (`arima-dev,m4_monthly.csv`): per-configuration error scores (e.g. MASE) obtained by cross-validating every ARIMA(p,d,q)(P,D,Q)[12] on each series.

The meta-learner maps features → configuration quality, so that at inference time we only need to fit a small shortlist instead of all 400 configurations.

### 1.1 Load metadata

In [ ]:
METADATA_DIR = Path("../assets/metadata/metaarima")

features = pd.read_csv(METADATA_DIR / "features-dev,m4_monthly.csv")
scores = pd.read_csv(METADATA_DIR / "arima-dev,m4_monthly.csv")

print(f"Features: {features.shape}")
print(f"Scores:   {scores.shape}")
features.head()

In [ ]:
# Merge features and scores on unique_id
id_col = "unique_id"
merged = scores.merge(features, on=id_col).set_index(id_col)

# Separate into X (features) and Y (per-config scores)
SEASON_LENGTH = 12

model_names = MetaARIMAUtils.get_models_sf(
    season_length=SEASON_LENGTH, max_config=ORDER_MAX, return_names=True
)

feature_cols = features.set_index(id_col).columns.tolist()
X = merged[feature_cols].fillna(-1)
Y = merged[[c for c in model_names if c in merged.columns]]

print(f"X (features):       {X.shape}")
print(f"Y (config scores):  {Y.shape}")
print(f"Configurations:     {len(model_names)}")

### 1.2 Create and train MetaARIMA

Key parameters:
- **`n_trials=25`** — shortlist 25 configurations per series at inference time.
- **`mmr_lambda=0.75`** — balance relevance (predicted score) and diversity (low correlation) in the shortlist. `1` = pure relevance, `0` = pure diversity.
- **`base_optim='halving'`** — use successive halving (instead of exhaustive search) to select the final ARIMA from the shortlist.

In [ ]:
catboost_params = BEST_CATBOOST_PARAMS["monthly"]

meta_arima = MetaARIMA(
    model=CatBoostRegressor(**catboost_params),
    freq="ME",
    season_length=SEASON_LENGTH,
    n_trials=25,
    quantile_thr=0.5,
    pca_n_components=100,
    mmr_lambda=0.75,
    base_optim="halving",
)

meta_arima.meta_fit(X, Y)

print(f"Meta-learner trained on {X.shape[0]} series.")
print(f"Number of target configurations: {len(meta_arima.model_names)}")

### 1.3 Save the trained model

In [ ]:
OUTPUT_DIR = Path("../assets/pretrained/metaarima")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "m4_monthly.joblib.gz"
meta_arima.save(str(output_path))

print(f"Saved to {output_path}")
print(f"File size: {output_path.stat().st_size / 1e6:.1f} MB")

---

## Part 2 — Inference on a New Series

Once trained (or loaded from disk), MetaARIMA acts like a regular forecasting model:
1. `.fit(df, freq, seas_length)` — extract features, shortlist configurations, select the best.
2. `.predict(h)` — forecast *h* steps ahead.

### 2.1 Load a pre-trained model

In [ ]:
meta = MetaARIMA.load(str(OUTPUT_DIR / "m4_monthly.joblib.gz"))

print(f"Loaded model: freq={meta.freq}, season_length={meta.season_length}")
print(f"Meta-learner fitted: {meta.is_fit}")
print(f"Shortlist size (n_trials): {meta.n_trials}")

### 2.2 Create an example time series

In [ ]:
np.random.seed(42)

n = 240  # 20 years of monthly data
t = np.arange(n)

# Trend + seasonality + noise
trend = 0.02 * t
seasonal = 3 * np.sin(2 * np.pi * t / 12)
noise = np.random.normal(0, 0.5, n)
y_values = 50 + trend + seasonal + noise

df = pd.DataFrame({
    "unique_id": ["example_series"] * n,
    "ds": pd.date_range("2005-01-01", periods=n, freq="ME"),
    "y": y_values,
})

df.tail()

In [ ]:
df.set_index("ds")["y"].plot(
    figsize=(12, 3), title="Example monthly series", ylabel="y"
);

### 2.3 Fit and forecast

In [ ]:
meta.fit(df, freq="ME", seas_length=12)

print(f"Selected configuration: {meta.selected_config}")

In [ ]:
# Point forecast
forecast = meta.predict(h=18)

forecast.head()

In [ ]:
# Forecast with 95% prediction intervals
forecast_ci = meta.predict(h=18, level=[95])

forecast_ci.head()

### 2.4 Visualise

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))

# Plot the last 60 observations + forecast
history = df.tail(60).set_index("ds")
ax.plot(history.index, history["y"], label="History", color="black")

fc = forecast_ci.set_index("ds")
ax.plot(fc.index, fc["MetaARIMA"], label="MetaARIMA forecast", color="tab:blue")
ax.fill_between(
    fc.index,
    fc["MetaARIMA-lo-95"],
    fc["MetaARIMA-hi-95"],
    alpha=0.2,
    color="tab:blue",
    label="95% interval",
)

ax.set_title(f"MetaARIMA forecast — selected config: {meta.selected_config}")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

### 2.5 Inspect the configuration shortlist

We can also look at the shortlist of ARIMA configurations that the meta-learner proposed before successive halving selected the final one.

In [ ]:
from metaforecast.coseal.metaarima._base import tsfeatures_uid

# Extract features for the example series
feat_df = tsfeatures_uid(df, freq=12)

print(f"Feature vector shape: {feat_df.shape}")
feat_df.T.head(10)

In [ ]:
# Get the configuration shortlist
shortlist = meta.meta_predict(feat_df)[0]

print(f"Shortlisted {len(shortlist)} configurations (n_trials={meta.n_trials}):")
for i, cfg in enumerate(shortlist, 1):
    marker = " <-- selected" if cfg == meta.selected_config else ""
    print(f"  {i:2d}. {cfg}{marker}")

---

## Summary

| Step | Method | Description |
|------|--------|-------------|
| Meta-train | `meta_arima.meta_fit(X, Y)` | Train the meta-learner from features + error scores |
| Save | `meta_arima.save(path)` | Serialise to disk |
| Load | `MetaARIMA.load(path)` | Restore from disk |
| Fit | `meta.fit(df, freq, seas_length)` | Shortlist + select best ARIMA for a new series |
| Predict | `meta.predict(h, level)` | Forecast with optional prediction intervals |